# exp001 baseline submission

First-pass Kaggle notebook baseline.

Plan:
- train on `high_confidence_candidates_rescued.csv`
- extract simple log-mel summary features
- fit a lightweight multiclass classifier
- score each 5-second test window
- write `submission.csv`

In [ ]:
from pathlib import Path
import json

import joblib
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tqdm.auto import tqdm

WORK_ROOT = Path.cwd().resolve()
LOCAL_ROOT = WORK_ROOT
if LOCAL_ROOT.name == "exp001_baseline":
    LOCAL_ROOT = LOCAL_ROOT.parents[1]

DATA_ROOT_CANDIDATES = [
    LOCAL_ROOT,
    Path("/kaggle/input/birdclef-2026"),
    Path("/kaggle/input/competitions/birdclef-2026"),
]
DATA_ROOT = next((path for path in DATA_ROOT_CANDIDATES if (path / "sample_submission.csv").exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError("Could not find BirdCLEF 2026 data. Expected sample_submission.csv in the local repo or Kaggle competition input mount.")

if (LOCAL_ROOT / "experiments" / "exp001_baseline").exists():
    EXPERIMENT_DIR = LOCAL_ROOT / "experiments" / "exp001_baseline"
else:
    EXPERIMENT_DIR = WORK_ROOT / "exp001_baseline"

ARTIFACT_DIR = EXPERIMENT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 32000
CHUNK_SECONDS = 5.0
N_MELS = 64

METADATA_CANDIDATES = [
    LOCAL_ROOT / "experiments" / "exp000_eda" / "high_confidence_candidates_rescued.csv",
    DATA_ROOT / "train.csv",
]
METADATA_PATH = next((path for path in METADATA_CANDIDATES if path.exists()), None)
if METADATA_PATH is None:
    raise FileNotFoundError("Could not find training metadata CSV.")

TRAIN_AUDIO_DIR = DATA_ROOT / "train_audio"
TEST_DIR = DATA_ROOT / "test_soundscapes"
SUBMISSION_TEMPLATE = DATA_ROOT / "sample_submission.csv"

{
    "work_root": str(WORK_ROOT),
    "data_root": str(DATA_ROOT),
    "experiment_dir": str(EXPERIMENT_DIR),
    "metadata_path": str(METADATA_PATH),
}


In [ ]:
available_test_audio = sorted(TEST_DIR.glob('*.ogg'))
print('test audio files found:', len(available_test_audio))
HAS_TEST_AUDIO = len(available_test_audio) > 0
if not HAS_TEST_AUDIO:
    print('No test .ogg files are visible in the current environment.')
    print('This can happen locally and in some Kaggle interactive runs for code competitions.')
    print('The notebook will still build a placeholder submission from sample_submission.csv.')


In [ ]:
def load_audio_chunk(path: Path, offset: float, duration: float, sample_rate: int = SAMPLE_RATE):
    info = sf.info(path)
    start_frame = int(offset * info.samplerate)
    num_frames = int(duration * info.samplerate)
    y, sr = sf.read(path, start=start_frame, frames=num_frames, dtype="float32")
    if getattr(y, "ndim", 1) == 2:
        y = y.mean(axis=1)
    if sr != sample_rate:
        y = librosa.resample(y, orig_sr=sr, target_sr=sample_rate)
        sr = sample_rate
    target_length = int(sample_rate * duration)
    if len(y) < target_length:
        y = np.pad(y, (0, target_length - len(y)))
    else:
        y = y[:target_length]
    return y, sr


def extract_features_from_audio(y: np.ndarray, sr: int, n_mels: int = N_MELS) -> np.ndarray:
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    return np.concatenate([
        log_mel.mean(axis=1),
        log_mel.std(axis=1),
        np.array([y.mean(), y.std(), float(np.max(np.abs(y)))])
    ]).astype(np.float32)


def extract_train_features(filename: str) -> np.ndarray:
    y, sr = load_audio_chunk(TRAIN_AUDIO_DIR / filename, offset=0.0, duration=CHUNK_SECONDS)
    return extract_features_from_audio(y, sr)


def parse_row_id(row_id: str) -> tuple[str, float]:
    stem, end_token = row_id.rsplit("_", 1)
    end_sec = float(end_token)
    return stem + ".ogg", max(0.0, end_sec - CHUNK_SECONDS)


def extract_test_features(row_id: str) -> np.ndarray:
    filename, offset = parse_row_id(row_id)
    y, sr = load_audio_chunk(TEST_DIR / filename, offset=offset, duration=CHUNK_SECONDS)
    return extract_features_from_audio(y, sr)

In [ ]:
train_df = pd.read_csv(METADATA_PATH)
sample_submission = pd.read_csv(SUBMISSION_TEMPLATE)

print("train rows", len(train_df))
print("train classes", train_df["primary_label"].nunique())
print("submission rows", len(sample_submission))
print("submission classes", len(sample_submission.columns) - 1)

In [ ]:
X_train = np.vstack([
    extract_train_features(filename)
    for filename in tqdm(train_df["filename"], desc="Training features")
])

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df["primary_label"])

model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
model.fit(X_train, y_train)

joblib.dump({
    "model": model,
    "label_encoder": label_encoder,
    "config": {
        "sample_rate": SAMPLE_RATE,
        "chunk_seconds": CHUNK_SECONDS,
        "n_mels": N_MELS,
    },
}, ARTIFACT_DIR / "kaggle_baseline_model.joblib")

print("trained classes", len(label_encoder.classes_))

In [ ]:
if HAS_TEST_AUDIO:
    test_features = np.vstack([
        extract_test_features(row_id)
        for row_id in tqdm(sample_submission["row_id"], desc="Test features")
    ])
    test_proba = model.predict_proba(test_features)
    print('test_proba shape', test_proba.shape)
else:
    test_proba = None
    print('Using placeholder submission values because no test audio is available in this run.')


In [ ]:
submission = sample_submission.copy()
default_priors = sample_submission.iloc[0, 1:].to_numpy(dtype=np.float32)
submission.iloc[:, 1:] = np.tile(default_priors, (len(submission), 1))

if test_proba is not None:
    class_to_submission_col = {col: idx for idx, col in enumerate(submission.columns[1:])}
    for model_idx, label in enumerate(label_encoder.classes_):
        if label in class_to_submission_col:
            submission.iloc[:, class_to_submission_col[label] + 1] = test_proba[:, model_idx]

submission.head()


In [ ]:
submission_path = WORK_ROOT / "submission.csv"
submission.to_csv(submission_path, index=False)
submission_path


In [ ]:
with open(ARTIFACT_DIR / "submission_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "metadata_path": str(METADATA_PATH),
        "sample_rate": SAMPLE_RATE,
        "chunk_seconds": CHUNK_SECONDS,
        "n_mels": N_MELS,
        "trained_classes": int(len(label_encoder.classes_)),
    }, f, indent=2)

print("Wrote submission.csv")